# 🔄 Agent-to-Agent: Hierarchical Research with Bigdata.com

This notebook shows a **hierarchical agent** that checks internal data first, then escalates to the **Bigdata.com Research Agent** for deep, cited research when needed.

## What This Demonstrates

**Internal-first flow:**
- **Primary Agent** queries internal DB (portfolios, holdings, transactions) and internal research (FAISS vector store).
- **Escalation** to Bigdata.com Research Agent only when the question needs external, multi-source analysis (20–60s, full citations).

**Research Agent:**
- Multi-step reasoning with RAG; answers include inline citations as source-name hyperlinks (no citation numbers).
- Retry, stream timeout, and logging (production-ready; see `research_client.py`).

**Framework flexibility:**
> This demo uses **LangChain** and **LangSmith**. The pattern—internal tools first, then one “research” tool—works with **CrewAI**, **AutoGen**, or custom graphs. The key is tool ordering and system prompt that prefers internal sources.

## Architecture

![Agent to Bigdata Research Agent](./static/agent_to_research_small.jpg)

**Key benefits:** Cost and latency control (internal answers are fast); full citations when escalating; reusable `langgraph_core` setup and tools.

## Use Cases

| Role | Example Questions |
|------|-------------------|
| **Equity Research** | Thesis validation, competitive analysis |
| **Credit Research** | Covenant analysis, refinancing risks |
| **Credit Risk** | Counterparty exposure, default drivers |

---

## Langsmith Tracing

![LangSmith Tracing](./static/langsmith.png)

## 1️⃣ Install Dependencies

> **Note:** If you're using `uv` to manage dependencies (recommended), you can skip the pip install cell below. Dependencies are already installed via `uv pip install -r requirements.txt`.

In [1]:
# Dependencies are installed via uv: uv pip install -r requirements.txt
%pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Import Libraries

Import LangChain, display helpers, and reusable setup/tools from `langgraph_core`. Display helpers: `display_query`, `display_tools_used`, `display_response`, `display_citations`.

In [2]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent as langchain_create_agent

# Reusable core: environment, data sources, tools, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    get_bigdata_tools,
    get_research_agent_tool,
    run_agent_query,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


## 3️⃣ Setup Environment

In [3]:
# Setup with LangSmith tracing
config = setup_environment(
    langsmith_project='bigdata-agent-to-agent',
    enable_tracing=True
)
print("\n🔗 View agent traces: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-to-agent
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...

🔗 View agent traces: https://smith.langchain.com


## 4️⃣ Initialize Data Sources

Create the internal database and vector store with sample financial data.

In [4]:
# Create SQLite database with portfolios, holdings, transactions
create_financial_database()
# Create vector store with internal research documents
create_vector_store()
print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")

✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)


## 5️⃣ Create Hierarchical Agent

This agent is configured to:
1. Check **internal sources first** (faster, proprietary data)
2. Use **quick external lookups** for company info
3. **Escalate** to Research Agent only when needed

System prompt and agent are defined below so the flow is transparent.

In [5]:
# System prompt: internal-first, then escalate to Research Agent; inline citations as hyperlinks
HIERARCHICAL_SYSTEM_PROMPT = """You are a senior financial research analyst with access to both internal company systems and the Bigdata.com Research Agent for external research.

**IMPORTANT: Follow this research hierarchy:**

1. **ALWAYS check internal sources FIRST:**
   - `internal_query_database` - Check our portfolio positions, transactions, and account data
   - `internal_portfolio_summary` - Get quick overview of specific portfolios
   - `internal_search_research` - Search our internal research documents, investment theses, and analyst notes

2. **For entity resolution (optional):**
   - `bigdata_lookup_company` - Get Bigdata entity IDs for companies (useful for identification)

3. **ESCALATE to Research Agent when internal sources are insufficient:**
   - `bigdata_research_agent` - Use when you need deep analysis, market-wide trends, current events, or information internal sources cannot answer. Research Agent takes 20-60 seconds but provides comprehensive analysis with citations.

**Decision Framework:** Portfolio/holdings → Internal DB first. Company we hold → Internal research first, escalate if needed. Company we don't hold / market trends / news → Research Agent.

**Citation format:** When using Research Agent output, use inline citations with **only the source name as a hyperlink**—no citation numbers. Format as markdown: [Source Name](url). The reader should see only clickable source names (e.g. [Nasdaq](url), [Yahoo! Finance](url)); do not add numbers like [1], [2] in the text. Do not add a separate "Sources" or "References" block at the end—inline citations are sufficient. For internal data, briefly mention "From internal database" or "According to internal research."

**Available Portfolios:** PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders). Check internal sources first, then escalate to Research Agent for external data.
**Do not offer suggestions for follow up questions**
"""

# Build tools: internal first, then lookup, then research agent
tools = (
    get_database_tools() +
    get_vectorstore_tools() +
    [t for t in get_bigdata_tools() if t.name == "bigdata_lookup_company"] +
    get_research_agent_tool()
)
llm = ChatOpenAI(model="gpt-5", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
agent = langchain_create_agent(llm, tools, system_prompt=HIERARCHICAL_SYSTEM_PROMPT)
tool_names = [t.name for t in tools]
print(f"✅ Hierarchical agent created with {len(tools)} tools:")
print(f"   Internal: {[n for n in tool_names if n.startswith('internal_')]}")
print(f"   External: {[n for n in tool_names if n.startswith('bigdata_')]}")

✅ Hierarchical agent created with 5 tools:
   Internal: ['internal_query_database', 'internal_portfolio_summary', 'internal_search_research']
   External: ['bigdata_lookup_company', 'bigdata_research_agent']


---

# 📈 Equity Research Use Cases

Questions that equity analysts typically ask.

### Query 1: Internal Holdings Check (No Escalation Expected)

Simple portfolio question - should use only internal tools.

In [6]:
query = """
What is our total exposure to NVIDIA across all portfolios? 
Include position sizes, average cost basis, and unrealized P&L.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_query_database: {
  "sql_query": "SELECT h.portfolio_id, p.portfolio_name, h.ticker, h.company_name, h.shares, h.avg_cost, h.current_price, h.market_value, h.unrealiz...
🔧 internal_query_database: {
  "sql_query": "SELECT h.portfolio_id, p.portfolio_name, a.currency\nFROM holdings h\nJOIN portfolios p ON h.portfolio_id = p.portfolio_id\nJOIN acc...


From internal database (USD):

By portfolio
- PF002 – AI & Semiconductor Focus
  - Shares: 12,000
  - Average cost: 450.00
  - Current price: 875.50
  - Market value: 10,506,000
  - Unrealized P&L: 5,106,000

- PF003 – Diversified Tech Leaders
  - Shares: 8,000
  - Average cost: 520.00
  - Current price: 875.50
  - Market value: 7,004,000
  - Unrealized P&L: 2,844,000

Totals across all portfolios
- Shares: 20,000
- Blended average cost: 478.00
- Market value: 17,510,000
- Unrealized P&L: 7,950,000

Note: No NVIDIA position in PF001.

### Query 2: Investment Thesis Validation (Internal + External)

Combines internal research with market data - may escalate for current news.

In [7]:
query = """
Review our investment thesis for NVIDIA:
1. What does our internal research say about NVIDIA's competitive position?
2. What recent market developments might affect this thesis?
3. Should we adjust our position based on current information?
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

🔧 internal_portfolio_summary: {
  "portfolio_id": "PF001"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF002"
}...
🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 internal_search_research: {
  "query": "NVIDIA NVDA competitive position thesis moat AI accelerators CUDA ecosystem networking InfiniBand NVLink supply chain HBM CoWoS Blackwel...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, investment-grade briefing on 2025\u20132026 developments that could affect an NVIDIA (NVDA) data-center AI thesis. Fo...


Here’s a concise review using our internal research first, then current external developments, and a positioning call.

1) Internal view of NVIDIA’s competitive position (from internal research)
- Leadership in AI accelerators: NVIDIA’s Hopper-to-Blackwell roadmap underpins a multiyear lead in training and increasingly inference; internal thesis (Dec 15, 2024) projected Blackwell B100/B200 at ~2.5x performance vs Hopper and sustained demand.
- Software moat: CUDA ecosystem (4M+ developers cited internally) and full-stack platform (CUDA, cuDNN, TensorRT, NCCL, NIMs) create high switching costs and time-to-solution advantages.
- Platform integration: NVLink/NVSwitch, InfiniBand networking, and DGX/NVL systems drive cluster-level performance and lock-in at hyperscalers and enterprises.
- Economics: Strong pricing power and mixed-shift to high-value data center SKUs; inference TAM expansion was a key upside driver in our thesis; risks flagged included China export controls, AMD competition, and supply constraints (HBM/CoWoS). Internal rating at last update: Strong Buy, PT $950. According to internal research.

2) Recent market developments that could affect the thesis (external, with citations)
- Blackwell/NVL72 ramp and adoption: Management highlighted broad GB200/NVL deployments across CSPs and model builders; investors are focused on Blackwell execution in 2025–2026 [Charlotte Observer](https://www.charlotteobserver.com/news/business/article314365118.html#storylink=partnerdigest_the), [Quartr Presentation Materials](https://files.quartr.com/conference-calls/2184a-2025-09-03-05-47-44.pdf?ref=UmF2ZW5QYWNr).
- Demand visibility: Commentary points to very strong Blackwell/Rubin order books into 2026, with large-scale AI infra spend tailwinds [Nasdaq](https://www.nasdaq.com/articles/nvidias-65-billion-forecast-sends-clear-message-about-ai-boom).
- Memory/packaging supply: HBM remains tight; reports indicate 2026 HBM3E contract prices up ~20% and increased HBM4 allocation to SK hynix, suggesting continued input cost strength and potential bottlenecks [Seeking Alpha](https://seekingalpha.com/news/4535511-samsung-sk-hynix-increase-hbm3e-prices-by-20-percent-for-2026-orders-report), [Yahoo! Finance](https://uk.finance.yahoo.com/news/samsung-electronics-posts-record-profit-000622725.html), [Yahoo! Finance](https://finance.yahoo.com/news/2-memory-stocks-buy-ubs-185020644.html).
- TSMC CoWoS capacity: Capacity expanding materially through 2026, but remains a gating factor for advanced AI accelerators [Nasdaq](https://www.nasdaq.com/articles/should-you-buy-tsm-while-its-under-400), [Taipei Times Online](https://www.taipeitimes.com/News/front/archives/2025/12/06/2003848408).
- Competitive dynamics: AMD’s MI355/MI400 Helios platform scheduled to ramp into 2026; street framing $14–15B AI revenue in 2026, implying AMD share gains but still ecosystem gap vs CUDA [AOL.com](https://www.aol.com/finance/amd-amd-q2-2025-earnings-150854690.html), [Yahoo! Finance](https://finance.yahoo.com/news/analyst-says-advanced-micro-devices-171220199.html). Intel Gaudi 3 improves perf/efficiency, but software traction remains the hurdle [Cisco](https://blogs.cisco.com/datacenter/accelerating-ethernet-native-ai-clusters-with-intel-gaudi-3-ai-accelerators-and-cisco-nexus-9000), [AOL.com](https://www.aol.com/finance/intel-eyeing-ai-acquisition-track-003500589.html).
- Hyperscaler custom silicon: Ongoing push (TPU/Trainium/Maia/MTIA) to diversify training/inference stacks; could pressure share/pricing at the margin while the overall pie grows [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-blackwell-vs-google-tpu-154032183.html).
- Export controls: Remain a swing factor; our internal risk work flagged meaningful China exposure risk. According to internal research.

3) Positioning recommendation (based on current information)
- We still support a core overweight. The moat (CUDA + systems + networking), Blackwell ramp, and enterprise inference monetization remain intact. According to internal research and corroborating market updates above.
- Risk management: Our exposure is concentrated. From internal database: PF002 has ~66% NVDA weight (10.51m of 15.99m MV), PF003 ~30% (7.00m of 23.23m), and combined across PF001–PF003 is ~35% of total MV in NVDA.
- Action:
  - Trim PF002 by 5–10% of NVDA holdings opportunistically into strength/events to reduce single-name risk while keeping a clear overweight.
  - Maintain PF003 near current weight; allow adds on verified Blackwell shipment/capacity upside or trims if AMD share gains/hyperscaler ASIC adoption accelerate.
  - Reallocate proceeds to adjacent beneficiaries (e.g., CoWoS/HBM supply chain) and maintain index/sector hedges given supply and policy risks. From internal database and according to internal research.

### Query 3: Competitive Analysis (Escalation Expected)

Deep competitive analysis requires external research agent.

In [ ]:
query = """
Perform a competitive analysis of AMD vs NVIDIA in the AI accelerator market:

1. First check: Do we have any internal research comparing these companies?
2. Then analyze: Current Market share trends, product roadmaps, and customer wins
3. Conclude: Which company is better positioned for 2025-2026?

Cite specific sources for your analysis.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# # display_citations(result)

🔧 internal_search_research: {
  "query": "AMD vs NVIDIA competitive analysis in AI accelerators (H100/H200/Blackwell vs MI300/MI325/MI350), CUDA vs ROCm, market share, customer w...
🔧 bigdata_research_agent: {
  "query": "Provide a concise, source-backed competitive analysis of AMD vs NVIDIA in the AI accelerator (data center GPU) market, focusing on 2024-...


---

# 💳 Credit Research Use Cases

Questions relevant to fixed income and credit analysts.

### Query 4: Debt and Leverage Analysis (Escalation Expected)

Credit-focused questions typically need external data.

In [ ]:
query = """
Analyze the credit profile of Intel Corporation:

1. Current debt levels and maturity schedule
2. Cash flow coverage ratios and liquidity position
3. Recent credit rating actions or outlook changes
4. Key risks that could affect their investment-grade status

Note: We don't hold Intel, so you may need external sources.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 5: Sector-Wide Credit Trends (Escalation Expected)

Macro credit analysis requires deep research.

In [ ]:
query = """
What are the key credit risks facing the semiconductor sector in the current environment?

Consider:
- Capital expenditure requirements and debt financing
- Cyclical demand patterns and inventory corrections
- Geopolitical risks (US-China tensions, export controls)
- Recent bond issuances or refinancing activity

Which semiconductor companies are most vulnerable from a credit perspective?
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 6: Refinancing Risk Assessment

In [ ]:
query = """
Which companies in our portfolios might face refinancing challenges in 2025-2026?

1. First, check what companies we hold across all portfolios
2. For each major holding, assess:
   - Debt maturity walls
   - Current interest coverage
   - Access to capital markets
3. Flag any companies with elevated refinancing risk
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

# ⚠️ Credit Risk Analyst Use Cases

Questions focused on counterparty risk and default analysis.

### Query 7: Counterparty Exposure Analysis (Internal First)

In [ ]:
query = """
Calculate our total counterparty exposure by company:

1. Sum up all positions across portfolios for each ticker
2. Calculate as percentage of total AUM
3. Identify our top 5 concentration risks
4. Flag any single-name exposures exceeding 15% of total

Present in a risk report format.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

### Query 8: ESG/Regulatory Risk (Escalation Expected)

In [ ]:
query = """
Assess regulatory and ESG risks for our technology holdings:

1. Which of our holdings face significant regulatory scrutiny?
2. What recent regulatory developments could impact valuations?
3. Are there any ESG controversies affecting our portfolio companies?
4. Which positions should we consider reducing due to regulatory risk?

Focus on antitrust, data privacy, and AI governance regulations.
"""
display_query(query)
result = run_agent_query(agent, query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

# 🔬 Custom Query

Try your own hierarchical research query:

In [ ]:
# Enter your own query
custom_query = """
Compare our internal research on NVIDIA with the latest market sentiment.
Is our thesis still valid?
"""
display_query(custom_query)
result = run_agent_query(agent, custom_query, verbose=True, return_tools=True)
display_tools_used(result)
display_response(result)
# display_citations(result)

---

## 📊 Observability

View detailed traces in LangSmith:
- Each query shows the **tool call sequence**
- See which tools were checked first vs escalated
- Monitor **latency** differences between internal vs external calls
- Track **token usage** for cost optimization

**Dashboard:** https://smith.langchain.com


## 🔟 Key Benefits of This Architecture

1. **Internal-first** — Fast answers from DB and vector store; escalate only when external research is needed.
2. **Full citations** — Research Agent returns inline [1], [2] and numbered sources; display via `display_citations()`.
3. **Production-ready** — Research client has retry, stream timeout, and full chat_id logging.
4. **Reusability** — `create_financial_database()`, `create_vector_store()`, and tools come from `langgraph_core.py`.
5. **Observability** — LangSmith traces show when the agent uses internal vs Research Agent tools.

## 🎯 Next Steps

- **Add Search API** — Combine with `get_bigdata_tools()` (Search + KG) for news/filings before escalating to Research Agent (see `agent_to_search.ipynb`).
- **Tune escalation** — Adjust system prompt so the agent escalates only for “deep research” or “recent market” questions.
- **Research effort** — Use `research_effort="lite"` for faster, lighter answers or `"standard"` for full depth.
- **Follow-up** — Use `result.chat_id` and `client.follow_up()` for multi-turn Research Agent conversations.
- **LangSmith** — Use traces to see tool order and Research Agent usage.

---

## 📚 Additional Resources

- **Bigdata.com API docs**: https://docs.bigdata.com
- **Research Agent client**: `Research_Agent_Sync_Response/README.md` (retry, logging, chat_id)
- **LangGraph / LangChain**: https://langchain-ai.github.io/langgraph/
- **LangSmith**: https://smith.langchain.com
- **Agent_To_BigData README**: `README.md` in this folder

**Questions?** support@bigdata.com